<a href="https://colab.research.google.com/github/khushijindal06/damwork/blob/main/damwork.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# DamWork: Duan RGB degradation with preserved bounding boxes

This notebook implements the required workflow:

```text
Clean Duan RGB image + original YOLO bounding box
                        |
                        v
haze / fog / low-light / rain-mist / mixed versions
                        |
                        v
degraded image + unchanged original bounding box
```

The degradation functions for haze, fog, low-light, and mixed conditions
are loaded from `DamVis/scripts/04_generate_degraded.py`. Rain-mist is
implemented in this notebook because it is not present in that script.

No resize, crop, flip, rotation, or perspective transform is used. Image
geometry remains unchanged, so the original bounding boxes remain valid.

## Important input requirement

Use clean Duan RGB images and machine-readable bounding-box files.
This notebook expects YOLO text labels:

```text
class_id x_center y_center width height
```

Coordinates must be normalized to `[0, 1]`.

Do **not** use the currently uploaded `UAV_piping_label_data/*.jpg` files
as clean source images. Their boxes and class names are already painted
into the pixels. Training images should remain clean; boxes are drawn only
for notebook previews.

## Colab setup

When opened from the badge above, Colab receives the notebook but not the repository files. The next cell clones `damwork` into `/content/damwork` and changes the working directory. For data stored in Google Drive, mount Drive and set `DUAN_IMAGES_DIR` and `DUAN_LABELS_DIR` in the configuration cell.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    colab_repo = Path("/content/damwork")
    if not colab_repo.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/khushijindal06/damwork.git",
                str(colab_repo),
            ],
            check=True,
        )
    os.chdir(colab_repo)
    print("Colab repository:", Path.cwd())
else:
    print("Local notebook directory:", Path.cwd())


In [ ]:
# Run this once in the Jupyter kernel if the packages are missing.
# %pip install opencv-python numpy pandas matplotlib tqdm noise

In [ ]:
import hashlib
import importlib.util
import json
import os
import shutil
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

## 1. Configure paths

Recommended source layout:

```text
Duan/
  images/
    image_001.jpg
  labels/
    image_001.txt
```

Images and labels may contain matching nested subfolders. Change the two
input paths below if your Duan dataset is stored elsewhere.

In [ ]:
def find_repo_root():
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "DamVis" / "scripts" / "04_generate_degraded.py").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Open this notebook from the damwork repository, or update REPO_ROOT."
    )


REPO_ROOT = find_repo_root()

CLEAN_IMAGES_DIR = Path(
    os.environ.get("DUAN_IMAGES_DIR", REPO_ROOT / "Duan" / "images")
)
YOLO_LABELS_DIR = Path(
    os.environ.get("DUAN_LABELS_DIR", REPO_ROOT / "Duan" / "labels")
)
OUTPUT_ROOT = Path(
    os.environ.get(
        "DUAN_OUTPUT_DIR",
        REPO_ROOT / "DamVis" / "dataset" / "duan_degraded",
    )
)

# Update these names to match the Duan class IDs.
CLASS_NAMES = {
    0: "piping",
}

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
JPEG_QUALITY = 92
GLOBAL_SEED = 2026
REQUIRE_LABELS = True
OVERWRITE = False

# Use a small integer while testing. Set to None for the complete dataset.
PROCESS_LIMIT = 5

# Keep False until the preview looks correct, then change to True.
RUN_BATCH = False

print("Repository:", REPO_ROOT)
print("Clean images:", CLEAN_IMAGES_DIR)
print("YOLO labels:", YOLO_LABELS_DIR)
print("Output:", OUTPUT_ROOT)

## 2. Load the existing DamVis degradation functions

In [ ]:
DAMVIS_SCRIPT = REPO_ROOT / "DamVis" / "scripts" / "04_generate_degraded.py"
if not DAMVIS_SCRIPT.exists():
    raise FileNotFoundError(f"Missing DamVis script: {DAMVIS_SCRIPT}")

spec = importlib.util.spec_from_file_location("damvis_degradation", DAMVIS_SCRIPT)
damvis = importlib.util.module_from_spec(spec)
spec.loader.exec_module(damvis)

print("Loaded:", DAMVIS_SCRIPT)
print("Existing families: haze, fog, low-light, mixed")

## 3. Add rain-mist degradation

In [ ]:
RAIN_MIST_CONFIGS = {
    "rain_mist_light": {
        "streak_density": 0.00018,
        "streak_alpha": 0.22,
        "mist_density": 0.10,
        "length": (8, 16),
        "angle_deg": (-12, -6),
        "thickness": 1,
    },
    "rain_mist_medium": {
        "streak_density": 0.00032,
        "streak_alpha": 0.34,
        "mist_density": 0.20,
        "length": (12, 24),
        "angle_deg": (-15, -7),
        "thickness": 1,
    },
    "rain_mist_heavy": {
        "streak_density": 0.00050,
        "streak_alpha": 0.48,
        "mist_density": 0.34,
        "length": (18, 34),
        "angle_deg": (-18, -8),
        "thickness": 2,
    },
}


def add_rain_mist(
    img_float,
    streak_density,
    streak_alpha,
    mist_density,
    length,
    angle_deg,
    thickness,
    rng,
):
    """Add slanted rain streaks and spatially varying valley mist."""
    height, width = img_float.shape[:2]

    rain_layer = np.zeros_like(img_float, dtype=np.float32)
    streak_count = max(1, int(height * width * streak_density))

    for _ in range(streak_count):
        x1 = int(rng.integers(0, width))
        y1 = int(rng.integers(0, height))
        streak_length = int(rng.integers(length[0], length[1] + 1))
        angle = np.deg2rad(rng.uniform(angle_deg[0], angle_deg[1]))
        x2 = int(np.clip(x1 + streak_length * np.sin(angle), 0, width - 1))
        y2 = int(np.clip(y1 + streak_length * np.cos(angle), 0, height - 1))
        brightness = float(rng.uniform(0.70, 1.00))
        cv2.line(
            rain_layer,
            (x1, y1),
            (x2, y2),
            (brightness, brightness, brightness),
            thickness=thickness,
            lineType=cv2.LINE_AA,
        )

    rain_layer = cv2.GaussianBlur(rain_layer, (3, 3), 0)
    rainy = np.clip(
        img_float * (1.0 - 0.06 * streak_alpha)
        + rain_layer * streak_alpha,
        0,
        1,
    )

    small_h = max(2, height // 16)
    small_w = max(2, width // 16)
    noise = rng.random((small_h, small_w)).astype(np.float32)
    noise = cv2.resize(noise, (width, height), interpolation=cv2.INTER_CUBIC)
    noise = cv2.GaussianBlur(noise, (0, 0), sigmaX=max(height, width) / 80)
    noise = (noise - noise.min()) / (np.ptp(noise) + 1e-8)

    valley_gradient = np.linspace(0.70, 1.15, height, dtype=np.float32)[:, None]
    mist_mask = np.clip(
        mist_density * (0.55 + 0.45 * noise) * valley_gradient,
        0,
        0.75,
    )
    mist_colour_bgr = np.array([0.94, 0.95, 0.96], dtype=np.float32)
    output = (
        rainy * (1.0 - mist_mask[:, :, None])
        + mist_colour_bgr * mist_mask[:, :, None]
    )
    return np.clip(output, 0, 1)

## 4. Define all 15 variants

In [ ]:
VARIANTS = {}

for name, params in damvis.HAZE_CONFIGS.items():
    VARIANTS[name] = {"family": "haze", "params": params}
for name, params in damvis.FOG_CONFIGS.items():
    VARIANTS[name] = {"family": "fog", "params": params}
for name, params in damvis.LOWLIGHT_CONFIGS.items():
    VARIANTS[name] = {"family": "lowlight", "params": params}
for name, params in RAIN_MIST_CONFIGS.items():
    VARIANTS[name] = {"family": "rain_mist", "params": params}
for name, params in damvis.MIXED_CONFIGS.items():
    VARIANTS[name] = {"family": "mixed", "params": params}

assert len(VARIANTS) == 15
pd.DataFrame(
    [
        {"variant": name, "family": item["family"], **item["params"]}
        for name, item in VARIANTS.items()
    ]
).fillna("")

## 5. Bounding-box and degradation helpers

In [ ]:
def stable_seed(*parts):
    text = "::".join(str(part) for part in parts)
    return int(hashlib.sha256(text.encode("utf-8")).hexdigest()[:8], 16)


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def load_yolo_boxes(label_path):
    boxes = []
    for line_number, raw_line in enumerate(
        label_path.read_text(encoding="utf-8").splitlines(), start=1
    ):
        line = raw_line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) != 5:
            raise ValueError(
                f"{label_path}:{line_number} must have 5 values, got {len(parts)}"
            )
        class_id = int(float(parts[0]))
        x_center, y_center, box_width, box_height = map(float, parts[1:])
        values = (x_center, y_center, box_width, box_height)
        if not all(0.0 <= value <= 1.0 for value in values):
            raise ValueError(
                f"{label_path}:{line_number} has coordinates outside [0, 1]"
            )
        if box_width <= 0 or box_height <= 0:
            raise ValueError(
                f"{label_path}:{line_number} has a non-positive box size"
            )
        boxes.append((class_id, *values))
    return boxes


def draw_yolo_boxes(img_bgr, boxes, colour=(0, 255, 0)):
    preview = img_bgr.copy()
    height, width = preview.shape[:2]
    for class_id, xc, yc, bw, bh in boxes:
        x1 = int((xc - bw / 2) * width)
        y1 = int((yc - bh / 2) * height)
        x2 = int((xc + bw / 2) * width)
        y2 = int((yc + bh / 2) * height)
        x1, x2 = sorted((np.clip(x1, 0, width - 1), np.clip(x2, 0, width - 1)))
        y1, y2 = sorted((np.clip(y1, 0, height - 1), np.clip(y2, 0, height - 1)))
        label = CLASS_NAMES.get(class_id, str(class_id))
        cv2.rectangle(preview, (x1, y1), (x2, y2), colour, 2)
        cv2.putText(
            preview,
            label,
            (x1, max(15, y1 - 5)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            colour,
            2,
            cv2.LINE_AA,
        )
    return preview


def apply_variant(img_float, variant_name, sample_key):
    item = VARIANTS[variant_name]
    params = item["params"]
    seed = stable_seed(GLOBAL_SEED, sample_key, variant_name)

    # Existing DamVis functions use NumPy's global random state.
    np.random.seed(seed)

    if item["family"] == "haze":
        return damvis.add_haze(img_float, params["beta"], params["A"])
    if item["family"] == "fog":
        return damvis.add_fog(
            img_float, params["density"], params["turbulence"]
        )
    if item["family"] == "lowlight":
        return damvis.add_low_light(
            img_float, params["gamma"], params["noise_sigma"]
        )
    if item["family"] == "mixed":
        return damvis.add_mixed(
            img_float,
            params["beta"],
            params["A"],
            params["gamma"],
            params["noise_sigma"],
        )
    if item["family"] == "rain_mist":
        return add_rain_mist(
            img_float,
            **params,
            rng=np.random.default_rng(seed),
        )
    raise KeyError(f"Unknown degradation family: {item['family']}")

## 6. Discover and validate Duan image-label pairs

In [ ]:
def discover_samples():
    if not CLEAN_IMAGES_DIR.exists():
        raise FileNotFoundError(
            f"Clean Duan image folder does not exist: {CLEAN_IMAGES_DIR}"
        )
    if REQUIRE_LABELS and not YOLO_LABELS_DIR.exists():
        raise FileNotFoundError(
            f"YOLO label folder does not exist: {YOLO_LABELS_DIR}"
        )

    image_paths = sorted(
        path
        for path in CLEAN_IMAGES_DIR.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )
    if PROCESS_LIMIT is not None:
        image_paths = image_paths[:PROCESS_LIMIT]

    samples = []
    missing_labels = []
    for image_path in image_paths:
        relative_image = image_path.relative_to(CLEAN_IMAGES_DIR)
        relative_label = relative_image.with_suffix(".txt")
        label_path = YOLO_LABELS_DIR / relative_label

        if not label_path.exists():
            missing_labels.append(relative_label)
            if REQUIRE_LABELS:
                continue
        else:
            load_yolo_boxes(label_path)

        samples.append(
            {
                "image_path": image_path,
                "label_path": label_path if label_path.exists() else None,
                "relative_image": relative_image,
                "relative_label": relative_label,
            }
        )

    if REQUIRE_LABELS and missing_labels:
        preview = "\n".join(str(path) for path in missing_labels[:10])
        raise FileNotFoundError(
            f"{len(missing_labels)} image(s) have no matching YOLO label. "
            f"First missing labels:\n{preview}"
        )
    if not samples:
        raise RuntimeError("No valid Duan image-label pairs were found.")
    return samples


# Run after updating the input paths.
# samples = discover_samples()
# print(f"Validated {len(samples)} image-label pairs")

## 7. Preview one clean image and five representative degradations

Bounding boxes are drawn only for display. Saved degraded images do not
contain painted boxes; the unchanged YOLO text files are saved separately.

In [ ]:
PREVIEW_VARIANTS = [
    "haze_medium",
    "fog_medium",
    "lowlight_dawn",
    "rain_mist_medium",
    "mixed_medium",
]


def preview_sample(sample):
    img_bgr = cv2.imread(str(sample["image_path"]))
    if img_bgr is None:
        raise ValueError(f"Could not read image: {sample['image_path']}")

    boxes = (
        load_yolo_boxes(sample["label_path"])
        if sample["label_path"] is not None
        else []
    )
    panels = [("clean", draw_yolo_boxes(img_bgr, boxes))]
    img_float = img_bgr.astype(np.float32) / 255.0

    for variant_name in PREVIEW_VARIANTS:
        degraded = apply_variant(
            img_float, variant_name, str(sample["relative_image"])
        )
        degraded_bgr = (degraded * 255).round().astype(np.uint8)
        panels.append(
            (variant_name, draw_yolo_boxes(degraded_bgr, boxes))
        )

    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    for axis, (title, panel_bgr) in zip(axes.flat, panels):
        axis.imshow(cv2.cvtColor(panel_bgr, cv2.COLOR_BGR2RGB))
        axis.set_title(title)
        axis.axis("off")
    plt.tight_layout()
    plt.show()


# Run after discovering samples:
# preview_sample(samples[0])

## 8. Generate the complete degraded dataset

In [ ]:
def save_jpeg(path, img_float):
    path.parent.mkdir(parents=True, exist_ok=True)
    image = (np.clip(img_float, 0, 1) * 255).round().astype(np.uint8)
    ok = cv2.imwrite(
        str(path),
        image,
        [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY],
    )
    if not ok:
        raise OSError(f"Failed to write image: {path}")


def run_generation(samples):
    image_root = OUTPUT_ROOT / "images"
    label_root = OUTPUT_ROOT / "labels"
    manifest_rows = []

    for sample in tqdm(samples, desc="Duan degradation"):
        img_bgr = cv2.imread(str(sample["image_path"]))
        if img_bgr is None:
            raise ValueError(f"Could not read image: {sample['image_path']}")
        original_shape = img_bgr.shape
        img_float = img_bgr.astype(np.float32) / 255.0
        label_hash = (
            sha256_file(sample["label_path"])
            if sample["label_path"] is not None
            else None
        )

        for variant_name, item in VARIANTS.items():
            relative_jpg = sample["relative_image"].with_suffix(".jpg")
            output_image = image_root / variant_name / relative_jpg
            output_label = label_root / variant_name / sample["relative_label"]

            if output_image.exists() and not OVERWRITE:
                degraded_bgr = cv2.imread(str(output_image))
                if degraded_bgr is None:
                    raise ValueError(f"Unreadable existing output: {output_image}")
            else:
                degraded = apply_variant(
                    img_float,
                    variant_name,
                    str(sample["relative_image"]),
                )
                save_jpeg(output_image, degraded)
                degraded_bgr = cv2.imread(str(output_image))

            if degraded_bgr.shape != original_shape:
                raise AssertionError(
                    f"Geometry changed for {output_image}: "
                    f"{original_shape} -> {degraded_bgr.shape}"
                )

            output_label_hash = None
            if sample["label_path"] is not None:
                output_label.parent.mkdir(parents=True, exist_ok=True)
                if OVERWRITE or not output_label.exists():
                    shutil.copy2(sample["label_path"], output_label)
                output_label_hash = sha256_file(output_label)
                if output_label_hash != label_hash:
                    raise AssertionError(
                        f"Bounding box changed for {output_label}"
                    )

            manifest_rows.append(
                {
                    "source_image": str(sample["image_path"]),
                    "source_label": (
                        str(sample["label_path"])
                        if sample["label_path"] is not None
                        else ""
                    ),
                    "variant": variant_name,
                    "family": item["family"],
                    "output_image": str(output_image),
                    "output_label": (
                        str(output_label)
                        if sample["label_path"] is not None
                        else ""
                    ),
                    "height": original_shape[0],
                    "width": original_shape[1],
                    "source_label_sha256": label_hash or "",
                    "output_label_sha256": output_label_hash or "",
                }
            )

    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    manifest = pd.DataFrame(manifest_rows)
    manifest.to_csv(OUTPUT_ROOT / "manifest.csv", index=False)

    run_metadata = {
        "clean_images": len(samples),
        "variants_per_image": len(VARIANTS),
        "generated_pairs": len(manifest),
        "geometry_changed": False,
        "bounding_boxes_copied_unchanged": True,
        "global_seed": GLOBAL_SEED,
        "variants": VARIANTS,
    }
    (OUTPUT_ROOT / "generation_metadata.json").write_text(
        json.dumps(run_metadata, indent=2),
        encoding="utf-8",
    )
    return manifest


if RUN_BATCH:
    samples = discover_samples()
    print(f"Generating {len(samples) * len(VARIANTS):,} image-label pairs")
    manifest = run_generation(samples)
    display(manifest.head())
else:
    print("RUN_BATCH is False. Validate paths and preview a sample first.")

## 9. Verify output geometry and bounding-box identity

In [ ]:
def verify_generation():
    manifest_path = OUTPUT_ROOT / "manifest.csv"
    if not manifest_path.exists():
        raise FileNotFoundError(
            f"No manifest found at {manifest_path}. Run generation first."
        )

    manifest = pd.read_csv(manifest_path, dtype=str).fillna("")
    expected_variants = set(VARIANTS)
    actual_variants = set(manifest["variant"])
    if actual_variants != expected_variants:
        raise AssertionError(
            f"Variant mismatch: expected {expected_variants}, got {actual_variants}"
        )

    label_rows = manifest[manifest["source_label"] != ""]
    changed_labels = label_rows[
        label_rows["source_label_sha256"]
        != label_rows["output_label_sha256"]
    ]
    if not changed_labels.empty:
        raise AssertionError(
            f"{len(changed_labels)} output labels differ from their originals"
        )

    missing_images = [
        path for path in manifest["output_image"] if not Path(path).exists()
    ]
    missing_labels = [
        path
        for path in label_rows["output_label"]
        if path and not Path(path).exists()
    ]
    if missing_images or missing_labels:
        raise FileNotFoundError(
            f"Missing outputs: {len(missing_images)} images, "
            f"{len(missing_labels)} labels"
        )

    summary = (
        manifest.groupby(["family", "variant"])
        .size()
        .rename("image_label_pairs")
        .reset_index()
    )
    print("All degraded images preserve original geometry.")
    print("All YOLO bounding-box files are byte-identical to the originals.")
    display(summary)
    return summary


# Run after generation:
# verification_summary = verify_generation()

## Output structure

```text
DamVis/dataset/duan_degraded/
  images/
    haze_light/
    haze_medium/
    haze_heavy/
    fog_light/
    fog_medium/
    fog_heavy/
    lowlight_dusk/
    lowlight_dawn/
    lowlight_night/
    rain_mist_light/
    rain_mist_medium/
    rain_mist_heavy/
    mixed_light/
    mixed_medium/
    mixed_heavy/
  labels/
    <the same 15 variant folders>
  manifest.csv
  generation_metadata.json
```

Every degraded image retains the source image dimensions. Its YOLO label
is copied byte-for-byte into the corresponding variant folder.